# Notebook 03 — Feature Engineering for Forecasting Models

This notebook creates powerful new features required for machine learning and deep learning forecasting models.

We will engineer:

🔹 Time-based features
	•	Year, Month, Week, Day, Weekday, IsWeekend
	•	Hour-of-day patterns (for hourly data)

🔹 Lag features
	•	Lag-1, lag-7, lag-30 (previous day / week / month)

🔹 Rolling window features
	•	7-day rolling mean
	•	30-day rolling mean
	•	Rolling standard deviation

🔹 Expanding window statistics
	•	Cumulative mean and cumulative sum

🔹 Seasonality encodings
	•	Fourier terms for yearly + weekly seasonality

🔹 Target encoding
	•	Weekly average by weekday
	•	Monthly average by month

🔹 Output

All processed datasets will be saved to: outputs/features/

## 🧩 Code Cell — Load Libraries & Data

In [1]:
import pandas as pd
import numpy as np
import os
from math import sin, cos, pi

# Load cleaned datasets from Notebook 01
daily = pd.read_csv("outputs/cleaned/daily_cleaned.csv")
weekly = pd.read_csv("outputs/cleaned/weekly_cleaned.csv")
monthly = pd.read_csv("outputs/cleaned/monthly_cleaned.csv")
hourly = pd.read_csv("outputs/cleaned/hourly_cleaned.csv")

# Convert date columns
daily['datum'] = pd.to_datetime(daily['datum'])
weekly['datum'] = pd.to_datetime(weekly['datum'])
monthly['datum'] = pd.to_datetime(monthly['datum'])
hourly['datum'] = pd.to_datetime(hourly['datum'])

print("Datasets loaded successfully.")

Datasets loaded successfully.


## 🧱 Helper Function — Fourier Features (Seasonality Encoding)

These are essential for models like XGBoost, LightGBM, Neural Networks.

In [2]:
def add_fourier_terms(df, column_date, period, order, prefix):
    """
    Adds Fourier series terms to model seasonality.
    period = cycle length (365 for year, 7 for weekly, 24 for hourly)
    order = number of sine/cosine components
    """
    t = np.arange(len(df))
    for k in range(1, order + 1):
        df[f"{prefix}_sin_{k}"] = np.sin(2 * np.pi * k * t / period)
        df[f"{prefix}_cos_{k}"] = np.cos(2 * np.pi * k * t / period)
    return df

## 🧩 Section 1 — Time-Based Features (Daily Data)

In [3]:
df = daily.copy()

df["Year"] = df["datum"].dt.year
df["Month"] = df["datum"].dt.month
df["Week"] = df["datum"].dt.isocalendar().week
df["Day"] = df["datum"].dt.day
df["Weekday"] = df["datum"].dt.weekday
df["IsWeekend"] = df["Weekday"].isin([5,6]).astype(int)

df.head()

,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,Hour,Weekday Name,Day,Weekday,Weekday_Name,Week,IsWeekend
0,2014-01-02,0.0,3.67,3.4,32.40,7.0,0.0,0.0,2.0,2014,1,248,Thursday,2,3,Thursday,1,0
1,2014-01-03,8.0,4.00,4.4,50.60,16.0,0.0,20.0,4.0,2014,1,276,Friday,3,4,Friday,1,0
2,2014-01-04,2.0,1.00,6.5,61.85,10.0,0.0,9.0,1.0,2014,1,276,Saturday,4,5,Saturday,1,1
3,2014-01-05,4.0,3.00,7.0,41.10,8.0,0.0,3.0,0.0,2014,1,276,Sunday,5,6,Sunday,1,1
4,2014-01-06,5.0,1.00,4.5,21.70,16.0,2.0,6.0,2.0,2014,1,276,Monday,6,0,Monday,2,0


## 🧩 Section 2 — Lag Features

We’ll generate lag features for the target category N02BE
(Change to other categories later if needed.)

In [4]:
target = "N02BE"

df[f"{target}_lag1"] = df[target].shift(1)     # previous day
df[f"{target}_lag7"] = df[target].shift(7)     # previous week
df[f"{target}_lag30"] = df[target].shift(30)   # previous month

## 🧩 Section 3 — Rolling Window Features

In [5]:
df[f"{target}_roll7"] = df[target].rolling(7).mean()
df[f"{target}_roll30"] = df[target].rolling(30).mean()
df[f"{target}_roll7_std"] = df[target].rolling(7).std()

## 🧩 Section 4 — Expanding Window Features

In [6]:
df[f"{target}_cummean"] = df[target].expanding().mean()
df[f"{target}_cumsum"] = df[target].expanding().sum()

## 🧩 Section 5 — Add Fourier Terms (Strong Seasonality)

Daily data → Yearly cycle = 365, Weekly cycle = 7

In [7]:
df = add_fourier_terms(df, "datum", period=365, order=3, prefix="yearly")
df = add_fourier_terms(df, "datum", period=7, order=3, prefix="weekly")

## 🧩 Section 6 — Target Encoding Features

Average sales per weekday

In [8]:
weekday_map = df.groupby("Weekday")[target].mean().to_dict()
df["Weekday_Avg"] = df["Weekday"].map(weekday_map)

Average sales per month

In [9]:
month_map = df.groupby("Month")[target].mean().to_dict()
df["Month_Avg"] = df["Month"].map(month_map)

## 🧹 Section 7 — Drop rows with missing values caused by lagging

In [10]:
df = df.dropna().reset_index(drop=True)
print("Final engineered dataset shape:", df.shape)
df.head()

Final engineered dataset shape: (2076, 40)


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,...,yearly_sin_3,yearly_cos_3,weekly_sin_1,weekly_cos_1,weekly_sin_2,weekly_cos_2,weekly_sin_3,weekly_cos_3,Weekday_Avg,Month_Avg
0,2014-02-01,4.33,4.32,5.0,43.0,13.0,1.0,14.0,0.0,2014,...,0.999769,0.021516,0.974928,-0.222521,-0.433884,-0.900969,-0.781831,0.623490,33.579719,36.13891
1,2014-02-02,7.00,3.00,0.2,13.5,6.0,2.0,8.0,0.0,2014,...,0.999546,-0.030120,0.433884,-0.900969,-0.781831,0.623490,0.974928,-0.222521,33.392859,36.13891
2,2014-02-03,5.00,1.00,8.5,32.4,16.0,1.0,1.0,0.0,2014,...,0.996659,-0.081676,-0.433884,-0.900969,0.781831,0.623490,-0.974928,-0.222521,29.232601,36.13891
3,2014-02-04,1.33,3.00,7.0,30.6,8.0,1.0,17.0,2.0,2014,...,0.991114,-0.133015,-0.974928,-0.222521,0.433884,-0.900969,0.781831,0.623490,28.373665,36.13891
4,2014-02-05,3.00,4.02,6.2,32.4,15.0,1.0,1.0,1.0,2014,...,0.982927,-0.183998,-0.781831,0.623490,-0.974928,-0.222521,-0.433884,-0.900969,28.074514,36.13891


## 💾 Section 8 — Save the processed dataset

In [11]:
os.makedirs("outputs/features", exist_ok=True)

df.to_csv("outputs/features/daily_features.csv", index=False)

print("Feature-engineered dataset saved successfully!")

Feature-engineered dataset saved successfully!
